# FisheriesAudit ALG — Entrega #07
# Red de Conflictos de Interés: CFP vs. Industria Pesquera

> **Autor:** Ariel Giamportone — Serie FisheriesAudit ALG 2026  
> **Fecha:** junio 2026  
> **Repositorio:** arielgiamportone/cfp-audit-intelligence

---

## Resumen ejecutivo

Este notebook analiza la superposición entre los **directores de empresas pesqueras** registrados en el Boletín Oficial de la República Argentina (Sección 4 — Sociedades) y las **personas que aparecen en las actas públicas del CFP** (Consejo Federal Pesquero) como votantes, asesores técnicos o delegados institucionales.

La hipótesis central: si un director de una empresa pesquera participa en la toma de decisiones sobre cuotas de captura que benefician a esa empresa, existe un potencial conflicto de interés que el marco regulatorio argentino (Ley 25.188, Art. 13) exige declarar y/o evitar.

> ⚠️ **Limitación:** Los datos de directivos son demostrativos (seed_demo).
> Requieren verificación contra el Boletín Oficial real antes de publicación.

## 1. Setup e importaciones

In [ ]:
import sys
from pathlib import Path

# Agregar src al path si se ejecuta desde notebooks/
repo_root = Path('.').resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import sqlite3

from src.acquisition.boletin_oficial_scraper import seed_cargos_demo, SEED_CARGOS_DEMO
from src.analysis.conflict_detector import ConflictDetector, NODE_PERSONA, NODE_EMPRESA

plt.style.use('dark_background')
pd.set_option('display.max_colwidth', 60)

DB_PATH = Path('../data/processed/catalog.db')
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f'BD: {DB_PATH.resolve()}')

## 2. Cargar datos de cargos directivos

Sembramos el dataset demo del Boletín Oficial y consultamos los registros disponibles.

In [ ]:
n_seed = seed_cargos_demo(DB_PATH)
print(f'Registros insertados (seed demo): {n_seed}')

cd = ConflictDetector(DB_PATH)
df_cargos = cd.get_cargos_directivos()
print(f'Total cargos directivos en BD: {len(df_cargos)}')
df_cargos.head(10)

### 2.1 Distribución de cargos por empresa

In [ ]:
empresas_count = df_cargos.groupby('empresa_nombre').size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
empresas_count.head(12).plot(kind='barh', ax=ax, color='#E65100')
ax.set_xlabel('N° de directivos registrados')
ax.set_title('Directivos por empresa pesquera (Boletín Oficial / seed demo)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('fig07_directivos_por_empresa.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada: fig07_directivos_por_empresa.png')

## 3. Detección de conflictos de interés

Cruzamos los directivos registrados con las menciones en actas CFP.

In [ ]:
conflicts = cd.detect_conflicts()
print(f'Conflictos detectados: {len(conflicts)}')

if not conflicts.empty:
    print('\nDistribución por severidad:')
    print(conflicts['severidad'].value_counts())
    print('\nDistribución por tipo:')
    print(conflicts['tipo_conflicto'].value_counts())
    display(conflicts.head(10))
else:
    print('Sin cruces detectados — ejecutar pipeline completo para datos reales.')
    print('Con datos demo seed, todos los conflictos serán tipo "potencial"')
    print('hasta que el pipeline CFP procese actas reales.')

### 3.1 Resumen ejecutivo del análisis

In [ ]:
summary = cd.conflict_summary(conflicts)

print('=== RESUMEN DE CONFLICTOS DE INTERÉS ===')
print(f'  Total conflictos detectados : {summary["n_total"]}')
print(f'  Severidad alta (🔴)          : {summary["n_alta"]}')
print(f'  Severidad media (🟡)         : {summary["n_media"]}')
print(f'  Severidad baja (🟢)          : {summary["n_baja"]}')

if summary['top_personas']:
    print('\nTop personas con mayor vinculación empresa-CFP:')
    for p in summary['top_personas'][:5]:
        print(f'  {p["persona_nombre"]} — {p["n_empresas"]} empresa(s)')

## 4. Construcción del grafo de conflictos

Grafo bipartito: **persona** (violeta) ↔ **empresa** (naranja).  
Grosor y color de arista según severidad del conflicto.

In [ ]:
# Si no hay conflictos reales, simulamos algunos para demostrar el grafo
if conflicts.empty:
    print('Generando conflictos sintéticos para demostración del grafo...')
    demo_rows = [
        {'persona_nombre': 'Jorge A. Suárez', 'empresa_nombre': 'CONARPESA',
         'cargo': 'socio_gerente', 'tipo_conflicto': 'potencial',
         'severidad': 'baja', 'n_resoluciones': 0, 'fuente_cargo': 'seed_demo', 'verificado': False},
        {'persona_nombre': 'Héctor N. Gutiérrez', 'empresa_nombre': 'ARGENOVA S.A.',
         'cargo': 'director_suplente', 'tipo_conflicto': 'participacion_decisiones',
         'severidad': 'media', 'n_resoluciones': 3, 'fuente_cargo': 'seed_demo', 'verificado': False},
        {'persona_nombre': 'Héctor N. Gutiérrez', 'empresa_nombre': 'CONARPESA',
         'cargo': 'socio_gerente', 'tipo_conflicto': 'voto_directo',
         'severidad': 'alta', 'n_resoluciones': 7, 'fuente_cargo': 'seed_demo', 'verificado': False},
        {'persona_nombre': 'Luis M. Soria', 'empresa_nombre': 'PESANTAR S.A.',
         'cargo': 'accionista', 'tipo_conflicto': 'participacion_decisiones',
         'severidad': 'media', 'n_resoluciones': 2, 'fuente_cargo': 'seed_demo', 'verificado': False},
        {'persona_nombre': 'Luis M. Soria', 'empresa_nombre': 'GLACIAR PESQUERA S.A.',
         'cargo': 'presidente', 'tipo_conflicto': 'potencial',
         'severidad': 'baja', 'n_resoluciones': 0, 'fuente_cargo': 'seed_demo', 'verificado': False},
        {'persona_nombre': 'Roberto D. Méndez', 'empresa_nombre': 'ESTREMAR S.A.',
         'cargo': 'director_suplente', 'tipo_conflicto': 'participacion_decisiones',
         'severidad': 'media', 'n_resoluciones': 4, 'fuente_cargo': 'seed_demo', 'verificado': False},
    ]
    conflicts = pd.DataFrame(demo_rows)
    print(f'Conflictos demo generados: {len(conflicts)}')

G = cd.build_conflict_graph(conflicts)
print(f'Grafo: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas')

### 4.1 Visualización del grafo

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

pos = nx.spring_layout(G, seed=42, k=3.0)

personas = [n for n, d in G.nodes(data=True) if d.get('tipo') == NODE_PERSONA]
empresas = [n for n, d in G.nodes(data=True) if d.get('tipo') == NODE_EMPRESA]

nx.draw_networkx_nodes(G, pos, nodelist=personas, node_color='#7B1FA2',
                       node_size=800, ax=ax, label='Personas (directivos)')
nx.draw_networkx_nodes(G, pos, nodelist=empresas, node_color='#E65100',
                       node_size=600, ax=ax, label='Empresas pesqueras')

for u, v, data in G.edges(data=True):
    sev = data.get('severidad', 'baja')
    color = {'alta': '#C62828', 'media': '#F57F17', 'baja': '#388E3C'}.get(sev, '#9E9E9E')
    width = {'alta': 4.0, 'media': 2.5, 'baja': 1.0}.get(sev, 1.0)
    nx.draw_networkx_edges(G, pos, edgelist=[(u, v)],
                           edge_color=color, width=width, ax=ax, alpha=0.8)

labels = {n: n[:20] for n in G.nodes()}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=7, ax=ax)

ax.set_title(
    'Red de Conflictos de Interés CFP — Industria Pesquera\n'
    '🟣 Personas con cargos directivos  🟠 Empresas pesqueras\n'
    'Aristas: 🔴 Alta  🟡 Media  🟢 Baja severidad',
    fontsize=11,
)
ax.legend(loc='upper left', fontsize=8)
ax.axis('off')
plt.tight_layout()
plt.savefig('fig07_red_conflictos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada: fig07_red_conflictos.png')

## 5. Análisis de centralidad

¿Qué personas y empresas son más centrales en la red de conflictos?

In [ ]:
if G.number_of_nodes() > 0:
    degree_centrality = nx.degree_centrality(G)
    betweenness = nx.betweenness_centrality(G)

    df_central = pd.DataFrame({
        'nodo': list(G.nodes()),
        'tipo': [G.nodes[n]['tipo'] for n in G.nodes()],
        'grado': [G.degree(n) for n in G.nodes()],
        'centralidad_grado': [round(degree_centrality[n], 3) for n in G.nodes()],
        'centralidad_intermediacion': [round(betweenness[n], 3) for n in G.nodes()],
    }).sort_values('centralidad_intermediacion', ascending=False)

    print('Top 10 nodos por centralidad de intermediación:')
    display(df_central.head(10))
else:
    print('Grafo vacío — sin datos de centralidad')

### 5.1 Componentes conexas y estructura de la red

In [ ]:
if G.number_of_nodes() > 0:
    n_comp = nx.number_connected_components(G)
    densidad = round(nx.density(G), 4)
    print(f'Componentes conexas: {n_comp}')
    print(f'Densidad del grafo: {densidad}')
    print(f'Nodos: {G.number_of_nodes()} (personas: {len(personas)}, empresas: {len(empresas)})')
    print(f'Aristas: {G.number_of_edges()}')

    # Componente más grande
    largest = max(nx.connected_components(G), key=len)
    print(f'Componente más grande: {len(largest)} nodos')
    print('  Integrantes:', ', '.join(sorted(largest)[:8]))

## 6. Tabla LaTeX para paper

Tabla de conflictos de alta y media severidad lista para incluir en artículo.

In [ ]:
alta_media = conflicts[conflicts['severidad'].isin(['alta', 'media'])].copy()

if alta_media.empty:
    print('Sin conflictos alta/media para tabla.')
else:
    latex = alta_media[['persona_nombre', 'empresa_nombre', 'cargo', 'severidad', 'n_resoluciones']].to_latex(
        index=False,
        caption='Conflictos de interés detectados (alta y media severidad). '  
                'Fuente: Boletín Oficial / actas CFP. Datos demo hasta verificación.',
        label='tab:conflictos_interes',
        column_format='llllr',
    )
    print(latex[:1500])

## 7. Conclusiones y próximos pasos

### Hallazgos preliminares

1. **Metodología replicable**: el cruce Boletín Oficial × Actas CFP detecta superposiciones persona-empresa de forma sistemática y trazable.

2. **Datos demo**: los 20 cargos directivos actuales son sintéticos. Con datos reales del BO y el pipeline CFP completo, la red tendrá poder analítico real.

3. **Estructura bipartita**: el grafo exhibe la arquitectura esperada en redes de captura regulatoria (regulatory capture): baja densidad, componentes pequeñas, pocos nodos de alta centralidad.

### Próximos pasos

1. **Scraping real del Boletín Oficial** → reemplazar seed_demo por datos verificados
2. **Pipeline CFP completo** → poblar menciones reales → conflictos de severidad alta
3. **Verificación legal** → experto revisa casos de alta severidad antes de publicación
4. **Análisis temporal** → ¿el conflicto existía en la fecha de la votación?
5. **Red extendida** → incluir familiares directos (mismo apellido) como nivel adicional

### Referencia bibliográfica

- Stigler, G.J. (1971). *The Theory of Economic Regulation.* Bell Journal of Economics.
- OCDE (2003). *Managing Conflict of Interest in the Public Service.*
- Ley 25.188 (Argentina): Ética en el Ejercicio de la Función Pública, Art. 13.
- FAO Code of Conduct 1995, Art. 7.1.2: transparencia en decisiones sobre recursos pesqueros.

> **Nota de reproducibilidad**: todos los resultados en este notebook se reproducen ejecutando  
> `jupyter nbconvert --to notebook --execute FisheriesAudit_ALG_07_conflictos_interes.ipynb`